## SCB Silver Layer Workforce Data

Transforms raw SCB JSON from the bronze volume into clean, typed Delta tables.

**Source**: `labour_market_platform.bronze_work_scb.landing/scb` (external volume)
**Target**: `labour_market_platform.silver_work_scb` (Delta tables)

| Silver Table | Source File(s) | Description |
|---|---|---|
| `ssyk_mapping` | `ssyk_2012_mapping.json` | SSYK 2012 occupation code to name reference |
| `wages` | `wages_*.json` | Wage distribution by sector, occupation, gender, year |
| `aku_employment` | `aku_employment_stock_occupation.json` | Employment stock by occupation, gender, year |
| `aku_population` | `aku_population_region_labourstatus.json` | Population by region, labour status, gender, year |
| `aku_unemployment` | `aku_unemployed_age_sex.json` | Unemployment by age group, gender, year |

**Transformations**:
* Flatten nested JSON -> typed columns
* Standardize and alias column names (snake_case)
* Cast metrics to `DOUBLE`; suppress SCB missing-value tokens as `NULL`
* Deduplicate overlapping partitions
* Enrich Wages and Employment tables with readable SSYK occupation names

In [0]:
# Imports
import json
import re
from glob import glob
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Configuration
catalog       = "labour_market_platform"
bronze_schema = "bronze_work_scb"
silver_schema = "silver_work_scb"
volume        = "landing"
bronze_root   = "scb"
base_path     = f"/Volumes/{catalog}/{bronze_schema}/{volume}/{bronze_root}"

#schema creation
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

def get_latest_partition(subfolder: str) -> str:
    """Returns the path of the most recently ingested partition in the bronze layer."""
    parts = sorted(glob(f"{base_path}/{subfolder}/ingestion_date=*"))
    if not parts:
        raise FileNotFoundError(f"No partitions found under {base_path}/{subfolder}/")
    return parts[-1]

print(f"Bronze root:{base_path}")
print(f"Latest wages partition: {get_latest_partition('wages')}")
print(f"Latest AKU partition: {get_latest_partition('aku')}")

In [0]:
def parse_scb_to_spark(file_paths: list, column_renames: dict = None) -> DataFrame:
    """Parses SCB JSON files into a flattened, typed PySpark DataFrame."""
    column_renames = column_renames or {}
    suppressed_vals = {"", ".", "..", "..."}

    def to_snake_case(s: str) -> str:
        return re.sub(r"[^a-z0-9]+", "_", s.lower().strip()).strip("_")

    all_rows     = []
    schema_order = {}

    for path in file_paths:
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)

        cols_meta = raw.get("columns", [])

        for col in cols_meta:
            raw_name   = to_snake_case(col["text"])
            final_name = column_renames.get(raw_name, raw_name)
            if final_name not in schema_order:
                col_type = StringType() if col["type"] in ("d", "t") else DoubleType()
                schema_order[final_name] = StructField(final_name, col_type, nullable=True)

        dim_names = [column_renames.get(to_snake_case(c["text"]), to_snake_case(c["text"])) for c in cols_meta if c["type"] in ("d", "t")]
        val_names = [column_renames.get(to_snake_case(c["text"]), to_snake_case(c["text"])) for c in cols_meta if c["type"] == "c"]

        for row in raw.get("data", []):
            record = {}
            for name, val in zip(dim_names, row["key"]):
                record[name] = val
            for name, val in zip(val_names, row["values"]):
                record[name] = None if val in suppressed_vals else float(val)
            all_rows.append(record)

    schema = StructType(list(schema_order.values()))
    return spark.createDataFrame(all_rows, schema=schema)

In [0]:
def save_to_silver(df: DataFrame, table_name: str, dedupe_columns: list) -> None:
    """Deduplicates and writes DataFrame as a Delta table."""
    df_clean     = df.dropDuplicates(dedupe_columns)
    target_table = f"{catalog}.{silver_schema}.{table_name}"

    df_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(target_table)
    
    print(f"Successfully wrote {df_clean.count()} rows to {target_table}")

In [0]:
ssyk_files = sorted(glob(f"{base_path}/reference/ingestion_date=*/ssyk_2012_mapping.json"))

with open(ssyk_files[-1], "r", encoding="utf-8") as f:
    ssyk_metadata = json.load(f)

ssyk_var = next(v for v in ssyk_metadata["variables"] if v["code"] == "Yrke2012")

df_ssyk = spark.createDataFrame(list(zip(ssyk_var["values"], ssyk_var["valueTexts"])), ["occupation_code", "raw_occupation_name"])
df_ssyk = df_ssyk.withColumn("occupation_name", F.trim(F.regexp_replace(F.col("raw_occupation_name"), r"^\d+\s+", ""))).drop("raw_occupation_name")

save_to_silver(df_ssyk, "ssyk_mapping", dedupe_columns=["occupation_code"])

# Create a 1-digit major group dataframe for AKU Employment joins
df_ssyk_major = df_ssyk.withColumn("major_code", F.substring(F.col("occupation_code"), 1, 1)) \
                       .dropDuplicates(["major_code"]) \
                       .select("major_code", F.col("occupation_name").alias("major_group_name"))

In [0]:
wages_renames = {
    "occupation_ssyk_2012": "occupation_code",
    "sex": "gender",
    "monthly_salary": "monthly_salary_avg",
    "10th_percentile": "salary_p10",
    "25th_percentile": "salary_p25",
    "75th_percentile": "salary_p75",
    "90th_percentile": "salary_p90"
}

wages_files = sorted(glob(f"{get_latest_partition('wages')}/*.json"))
df_wages = parse_scb_to_spark(wages_files, wages_renames)

df_wages = df_wages.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_wages = df_wages.replace({"0": "all sectors", "1": "central government", "2": "municipalities", "3": "county council", "1-3": "public sector", "4-5": "private sector"}, subset=["sector"])

# Enrich with SSYK
df_wages = df_wages.join(df_ssyk, on="occupation_code", how="left")

save_to_silver(df_wages, "wages", dedupe_columns=["sector", "occupation_code", "gender", "year"])

In [0]:
emp_renames = {
    "degree_of_attachment_to_the_labour_market": "attachment_code",
    "occupation": "occupation_code",
    "sex": "gender",
    "employed_persons": "employed_thousands"
}

emp_files = sorted(glob(f"{get_latest_partition('aku')}/aku_employment_stock_occupation*.json"))
df_emp    = parse_scb_to_spark(emp_files, emp_renames)

df_emp = df_emp.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_emp = df_emp.replace({"ANSTTOT": "employees total", "FA": "permanent employees", "SYSTOT": "total employment"}, subset=["attachment_code"])

# Enrich with SSYK (Using 1-digit major group)
df_emp = df_emp.join(df_ssyk_major, df_emp.occupation_code == df_ssyk_major.major_code, how="left").drop("major_code")

save_to_silver(df_emp, "aku_employment", dedupe_columns=["attachment_code", "occupation_code", "gender", "year"])

In [0]:
#population renames
pop_renames = {
    "labour_status":     "labour_status_code",
    "sex": "gender",
    "thousands":              "pop_thousands",
    "margin_of_error_1000s":  "moe_thousands",
    "percent":                  "pop_percent",
    "margin_of_error_percent":  "moe_percent",
}

#mapping regions
region_mapping = {
    "00": "Sweden",
    "0050": "Sweden excl. Stockholm, Göteborg, Malmö",
    "01": "Stockholm",
    "0180": "Stockholm",   
    "03": "Uppsala",
    "04": "Södermanland",
    "05": "Östergötland",
    "06": "Jönköping",
    "07": "Kronoberg",
    "08": "Kalmar",
    "09": "Gotland",
    "10": "Blekinge",
    "12": "Skåne",
    "1280": "Skåne",
    "13": "Halland",
    "14": "Västra Götaland",
    "1480": "Västra Götaland",
    "17": "Värmland",
    "18": "Örebro",
    "19": "Västmanland",
    "20": "Dalarna",
    "21": "Gävleborg",
    "22": "Västernorrland",
    "23": "Jämtland",
    "24": "Västerbotten",
    "25": "Norrbotten",
}

pop_files = sorted(glob(f"{get_latest_partition('aku')}/aku_population_region_labourstatus*.json"))
df_pop    = parse_scb_to_spark(pop_files, pop_renames)

df_pop = df_pop.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_pop = df_pop.replace(region_mapping, subset=["region"])
df_pop = df_pop.replace({
    "TOTALT": "total",
    "ALÖS": "unemployed",
    "EIAKR": "not in labour force",
    "SYS": "employed",
}, subset=["labour_status_code"])

save_to_silver(df_pop, "aku_population", dedupe_columns=["region", "labour_status_code", "gender", "year"])

In [0]:
# PROCESS AKU UNEMPLOYMENT
unemp_renames = {
    "labour_status":        "labour_status_code",
    "age":                           "age_group",
    "sex": "gender",
    "1000s":                   "unemp_thousands",
    "margin_of_error_1000s":     "moe_thousands",
    "percent":                   "unemp_percent",
    "margin_of_error_percent":     "moe_percent",
}

unemp_files = sorted(glob(f"{get_latest_partition('aku')}/aku_unemployed_age_sex*.json"))
df_unemp    = parse_scb_to_spark(unemp_files, unemp_renames)

df_unemp = df_unemp.replace({"1": "men", "2": "women", "1+2": "total"}, subset=["gender"])
df_unemp = df_unemp.replace({"ALÖS": "unemployed"}, subset=["labour_status_code"])

df_unemp = df_unemp.filter(F.col("age_group").isin("15-24", "tot15-74"))

save_to_silver(df_unemp, "aku_unemployment", dedupe_columns=["labour_status_code", "age_group", "gender", "year"])

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.ssyk_mapping

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.wages LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_employment LIMIT 10

In [0]:
%sql
SELECT DISTINCT region FROM labour_market_platform.silver_work_scb.aku_population

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_population WHERE Region NOT IN ("Sweden", "Sweden excl. Stockholm, Göteborg, Malmö") LIMIT 10

In [0]:
%sql
SELECT * FROM labour_market_platform.silver_work_scb.aku_unemployment LIMIT 10